In [3]:
import os
import numpy as np
from scipy.io.wavfile import write, read
from python_speech_features import mfcc
from hmmlearn.hmm import GaussianHMM

# Create small sample audio dataset
sample_rate = 16000
duration = 1
t = np.linspace(0, duration, sample_rate)

words = {
    "hello": 440,
    "yes": 660,
    "no": 880
}

os.makedirs("speech_data", exist_ok=True)

for word, freq in words.items():
    folder = os.path.join("speech_data", word)
    os.makedirs(folder, exist_ok=True)

    for i in range(5):
        signal = np.sin(2 * np.pi * freq * t)
        noise = 0.02 * np.random.randn(len(t))
        audio = signal + noise
        write(
            os.path.join(folder, f"{word}_{i}.wav"),
            sample_rate,
            (audio * 32767).astype(np.int16)
        )

print("Sample speech dataset created.")

Sample speech dataset created.


In [4]:
import os
import numpy as np
from scipy.io.wavfile import read
from python_speech_features import mfcc
from hmmlearn.hmm import GaussianHMM

class ModelHMM:
    def __init__(self, num_components=4, num_iter=1000):
        self.n_components = num_components
        self.n_iter = num_iter
        self.models = {}

    def train(self, training_data, label):
        model = GaussianHMM(
            n_components=self.n_components,
            covariance_type="diag",
            n_iter=self.n_iter
        )
        model.fit(training_data)
        self.models[label] = model

    def predict(self, input_data):
        scores = {}
        for label, model in self.models.items():
            scores[label] = model.score(input_data)
        return max(scores, key=scores.get), scores

speech_model = ModelHMM()

input_folder = "speech_data"

for label in os.listdir(input_folder):
    folder = os.path.join(input_folder, label)

    if not os.path.isdir(folder):
        continue

    features = []

    for filename in os.listdir(folder):
        if filename.endswith(".wav"):
            filepath = os.path.join(folder, filename)
            sampling_freq, signal = read(filepath)
            mfcc_features = mfcc(signal, sampling_freq)
            features.append(mfcc_features)

    training_data = np.vstack(features)
    speech_model.train(training_data, label)

print("HMM speech models trained successfully.")

# Test one file
test_file = "speech_data/hello/hello_0.wav"
sampling_freq, signal = read(test_file)
test_features = mfcc(signal, sampling_freq)

predicted_word, scores = speech_model.predict(test_features)

print("Test file:", test_file)
print("Predicted word:", predicted_word)
print("Scores:", scores)

HMM speech models trained successfully.
Test file: speech_data/hello/hello_0.wav
Predicted word: hello
Scores: {'no': -38449.73438939006, 'hello': -2510.640731324009, 'yes': -44266.017156558046}
